# Differential Gene Expression Analysis — UPS (26 Samples)

## Description

This notebook uses DESeq2 to compare gene expression between C1 (21 samples) and C2 (5 samples) in UPS.

Five samples were removed from the original 31 samples.

## Requirements

### Input Data

- `rsem.merged.gene_counts.tsv`: Raw RNA-seq gene count data.

### Required Software

- R with Jupyter Notebook (R kernel)
- DESeq2
- apeglm
- tidyverse

### Parameters

- Remove 5 samples from the original dataset.
- Keep genes with counts ≥ 30 in at least 5 samples.
- Compare C2 with C1 (C1 as the reference).
- Use `padj < 0.05` to identify significant genes.
- Use apeglm for log2 fold change shrinkage.

## Results

### 1. What important information do I gain from this analysis?

- Identify significantly differentially expressed genes between C1 and C2.
- Identify genes with higher expression in C1 or C2.

### 2. What plots, files etc. are created and where?

Three TSV files are saved in `../results/deseq2/`:

- `DESeq2_all_26samples.tsv`: Results for all tested genes.
- `C1_higher_26samples.tsv`: Significant genes with higher expression in C1.
- `C2_higher_26samples.tsv`: Significant genes with higher expression in C2.

No plots are generated in this notebook.

In [27]:
# Differentialexpression


In [28]:
# Load DESeq2
library(DESeq2)

In [29]:
# Set input file paths
data_location = '~/Documents/ups/data/rnaseq/rsem/rsem.merged.gene_counts.tsv'

In [30]:
# Load raw gene count data
counts_raw <- read.delim(data_location, check.names = FALSE)
# Preview the data
head(counts_raw)

,gene_id,gene_name,PDXSAR008,PDXSAR010,PDXSAR023,PDXSAR028,SAR005,SAR007,SAR008,SAR009,⋯,SAR055,SAR057,SAR058,SAR061,SAR066,SAR074,SAR083,SAR084,SAR085,SAR086
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000000003,TSPAN6,346,53.00,1019,601.00,99.00,30,131,94.00,⋯,103.00,131,392.00,130.00,113,216.99,131.00,115,492.00,189.00
2,ENSG00000000005,TNMD,0,0.00,360,0.00,0.00,0,0,0.00,⋯,0.00,1,0.00,3.00,3,1.00,0.00,0,5.00,0.00
3,ENSG00000000419,DPM1,86,100.01,65,96.00,15.00,21,62,49.99,⋯,72.00,94,96.99,59.99,27,70.00,23.99,43,69.00,49.99
4,ENSG00000000457,SCYL3,133,105.00,110,126.00,51.99,40,72,87.01,⋯,53.00,78,40.00,45.01,46,45.00,55.00,38,46.99,49.01
5,ENSG00000000460,FIRRM,244,198.00,222,123.01,38.99,23,144,24.00,⋯,48.01,199,51.00,56.01,23,58.00,53.00,18,21.99,44.00
6,ENSG00000000938,FGR,1,427.01,0,5.00,84.00,29,7,76.00,⋯,0.00,13,33.00,12.00,16,21.00,17.00,28,15.00,4.00


In [31]:
# Load packages for gene annotation and data manipulation
library(tidyverse)

In [32]:
# Prepare count matrix for DESeq2
count_matrix <- counts_raw %>%
    select(-gene_name) %>%
    column_to_rownames("gene_id") %>%
    as.matrix()
# Convert counts to integers
count_matrix <- round(count_matrix)

In [33]:
# Exclude 5 outlier samples
exclude_samples <- c(
    "PDXSAR010",
    "PDXSAR023",
    "SAR023",
    "SAR050",
    "SAR009"
)

count_matrix <- count_matrix[
    ,
    !(colnames(count_matrix) %in% exclude_samples)
]

In [34]:
# Define Cluster 2 samples
cluster2_samples <- c(
    "PDXSAR008",
    "SAR008",
    "PDXSAR028",
    "SAR028",
    "SAR055"
)

In [35]:
# Assign samples to C1 or C2
sample_info <- data.frame(
    sample = colnames(count_matrix),
    cluster = ifelse(
        colnames(count_matrix) %in% cluster2_samples,
        "C2",
        "C1"
    )
)
# Use sample names as row names
rownames(sample_info) <- sample_info$sample
# Check the number of samples in each cluster
table(sample_info$cluster)


C1 C2 
21  5 

In [36]:
# Set C1 as the reference level
sample_info$cluster <- factor(
    sample_info$cluster,
    levels = c("C1", "C2")
)
# Create DESeq2 dataset
dds <- DESeqDataSetFromMatrix(
    countData = count_matrix,
    colData = sample_info,
    design = ~ cluster
)

converting counts to integer mode



In [37]:
# Filter low-count genes
smallestGroupSize <- 5

keep <- rowSums(counts(dds) >= 30) >= smallestGroupSize

dds <- dds[keep, ]
# Check the dimensions after filtering
dim(dds)

[1] 12945    26

In [38]:
# Run DESeq2 analysis
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 303 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [39]:
# Check available DESeq2 result names
resultsNames(dds)

[1] "Intercept"        "cluster_C2_vs_C1"

In [40]:
# Extract differential expression results for C2 vs C1
res <- results(
    dds,
    name = "cluster_C2_vs_C1")
# Shrink log2 fold changes using apeglm
res <- lfcShrink(
    dds,
    coef = "cluster_C2_vs_C1",
    res = res,
    type = "apeglm"
)

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



In [41]:
exists("dds")

[1] TRUE

In [42]:
# Filter and sort significant DEGs
get_hits <- function(deseq_result){
    res_df <- as.data.frame(deseq_result)
# Keep significant genes
    filtered_sorted_res <- res_df[
        (!is.na(res_df$padj)) & (res_df$padj < 0.05),
    ]
# Count up- and down-regulated genes
    n_C2 <- sum(filtered_sorted_res$log2FoldChange > 0)
    n_C1 <- sum(filtered_sorted_res$log2FoldChange < 0)

    message(
        n_C1, " upregulated and ",
        n_C2, " downregulated hits (padj < 0.05)"
    )
   return(filtered_sorted_res)
}

In [43]:
get_hits(res) %>% head(n=10)

1592 upregulated and 547 downregulated hits (padj < 0.05)



,baseMean,log2FoldChange,lfcSE,pvalue,padj
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000000971,829.35788,-1.6362490,0.8059955,7.284929e-04,8.995000e-03
ENSG00000001617,88.89466,-1.1633308,0.6004494,1.986496e-03,1.802049e-02
ENSG00000002933,76.68261,-2.6468129,0.8655109,2.763226e-05,9.428533e-04
ENSG00000003096,69.29380,-2.0753026,0.9984478,4.470359e-04,6.334322e-03
ENSG00000003137,59.14833,-2.5128111,0.8199566,3.433360e-05,1.072175e-03
ENSG00000003400,83.35526,-2.0229029,0.6847050,6.075186e-05,1.557039e-03
ENSG00000003402,513.46335,-1.3296992,0.3396644,4.219154e-06,2.360158e-04
ENSG00000003436,668.90716,-2.7160346,0.5483572,1.065437e-08,3.115434e-06
ENSG00000004399,1206.95994,-1.2432767,0.4527598,2.559163e-04,4.304078e-03


In [44]:
# Preview top 10 C1-higher significant genes
get_hits(res) %>% head(n = 10)

1592 upregulated and 547 downregulated hits (padj < 0.05)



,baseMean,log2FoldChange,lfcSE,pvalue,padj
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000000971,829.35788,-1.6362490,0.8059955,7.284929e-04,8.995000e-03
ENSG00000001617,88.89466,-1.1633308,0.6004494,1.986496e-03,1.802049e-02
ENSG00000002933,76.68261,-2.6468129,0.8655109,2.763226e-05,9.428533e-04
ENSG00000003096,69.29380,-2.0753026,0.9984478,4.470359e-04,6.334322e-03
ENSG00000003137,59.14833,-2.5128111,0.8199566,3.433360e-05,1.072175e-03
ENSG00000003400,83.35526,-2.0229029,0.6847050,6.075186e-05,1.557039e-03
ENSG00000003402,513.46335,-1.3296992,0.3396644,4.219154e-06,2.360158e-04
ENSG00000003436,668.90716,-2.7160346,0.5483572,1.065437e-08,3.115434e-06
ENSG00000004399,1206.95994,-1.2432767,0.4527598,2.559163e-04,4.304078e-03


In [45]:
# Add gene IDs and gene names to significant DEGs
hits <- get_hits(res)%>%
    rownames_to_column("gene_id") %>%
    left_join(counts_raw %>% select(gene_id, gene_name),by = "gene_id")
# Preview results
head(hits, 10)

1592 upregulated and 547 downregulated hits (padj < 0.05)



,gene_id,baseMean,log2FoldChange,lfcSE,pvalue,padj,gene_name
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,ENSG00000000971,829.35788,-1.6362490,0.8059955,7.284929e-04,8.995000e-03,CFH
2,ENSG00000001617,88.89466,-1.1633308,0.6004494,1.986496e-03,1.802049e-02,SEMA3F
3,ENSG00000002933,76.68261,-2.6468129,0.8655109,2.763226e-05,9.428533e-04,TMEM176A
4,ENSG00000003096,69.29380,-2.0753026,0.9984478,4.470359e-04,6.334322e-03,KLHL13
5,ENSG00000003137,59.14833,-2.5128111,0.8199566,3.433360e-05,1.072175e-03,CYP26B1
6,ENSG00000003400,83.35526,-2.0229029,0.6847050,6.075186e-05,1.557039e-03,CASP10
7,ENSG00000003402,513.46335,-1.3296992,0.3396644,4.219154e-06,2.360158e-04,CFLAR
8,ENSG00000003436,668.90716,-2.7160346,0.5483572,1.065437e-08,3.115434e-06,TFPI
9,ENSG00000004399,1206.95994,-1.2432767,0.4527598,2.559163e-04,4.304078e-03,PLXND1


In [46]:
# Get C2-higher significant genes
hits_C2 <- hits %>%
filter(log2FoldChange > 0) %>%
arrange(desc(log2FoldChange))
# Preview top 10 C2-higher genes
head(hits_C2, 10)

,gene_id,baseMean,log2FoldChange,lfcSE,pvalue,padj,gene_name
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,ENSG00000153132,53.51254,3.847183,1.0475521,5.411400e-05,1.457258e-03,CLGN
2,ENSG00000162490,48.97897,3.357543,1.0205757,1.833809e-04,3.418845e-03,DRAXIN
3,ENSG00000188064,41.20703,3.355242,1.1627900,6.676518e-04,8.421576e-03,WNT7B
4,ENSG00000168528,69.17975,3.221788,0.7365632,2.322000e-06,1.464454e-04,SERINC2
5,ENSG00000167723,26.32981,3.164442,0.7107265,1.572630e-06,1.124081e-04,TRPV3
6,ENSG00000168621,54.90486,2.853523,0.8505995,1.253617e-04,2.639777e-03,GDNF
7,ENSG00000271127,31.04805,2.836567,0.9609270,4.536317e-04,6.406614e-03,ENSG00000271127
8,ENSG00000142627,142.45525,2.745442,0.5970732,7.122896e-07,6.150549e-05,EPHA2
9,ENSG00000163814,315.30364,2.613645,0.6811117,1.935979e-05,7.240787e-04,CDCP1


In [47]:
# Separate significant DEGs into C1-higher and C2-higher genes
hits_C1 <- hits %>%
    filter(log2FoldChange < 0) %>%
    arrange(log2FoldChange)

hits_C2 <- hits %>%
    filter(log2FoldChange > 0) %>%
    arrange(desc(log2FoldChange))

In [48]:
# Check the dimensions of C1- and C2-higher gene sets
dim(hits_C1)
dim(hits_C2)

[1] 1592    7

[1] 547   7

In [51]:
# Save C1- and C2-higher genes for the 26-sample analysis
write.table(
    hits_C1,
    file = "../results/deseq2/C1_higher_26samples.tsv",
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)

write.table(
    hits_C2,
    file = "../results/deseq2/C2_higher_26samples.tsv",
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)

In [50]:
# Export the COMPLETE DESeq2 result table for a volcano plot
# This is the same 26-sample DESeq2 run (C1 reference, C2 vs C1).
stopifnot(ncol(count_matrix) == 26)
stopifnot(sum(sample_info$cluster == "C2") == 5)

res_all <- as.data.frame(res) %>%
    tibble::rownames_to_column("gene_id") %>%
    dplyr::left_join(
        counts_raw %>% dplyr::select(gene_id, gene_name),
        by = "gene_id"
    )

output_file <- "../results/deseq2/DESeq2_all_26samples.tsv"
write.table(
    res_all,
    file = output_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)
cat("Saved:", normalizePath(output_file), "\n")
cat("Number of genes:", nrow(res_all), "\n")

Saved: /Users/ziyao/Documents/ups/analysis/rnaseq/results/deseq2/DESeq2_all_26samples.tsv 
Number of genes: 12945 
